# GHSL Population Export — India
Export GHS_POP (P2023A) for India across all available epochs (1975–2020) to Google Drive at 100 m resolution in WGS84.

In [3]:
import ee

# Authenticate (only needed once per environment)
ee.Authenticate()
ee.Initialize()

In [4]:
# ── Study area ────────────────────────────────────────────────────────────────
India = (ee.FeatureCollection('USDOS/LSIB_SIMPLE/2017')
           .filter(ee.Filter.eq('country_na', 'India')))

# ── GHSL P2023A available epochs ──────────────────────────────────────────────
years = [ 2015, 2020, 2025]

In [4]:
# ── Loop: clean and export each year ─────────────────────────────────────────
for year in years:
    # 1. Load image
    raw = ee.Image(f'JRC/GHSL/P2023A/GHS_POP/{year}')

    # 2. Select population band, clip, clamp negatives to 0
    clean = (
        raw
        .select('population_count')
        .clip(India)
        .max(ee.Image(0))  # removes all negative/NoData artefacts
    )

    # 3. Export to Drive
    task = ee.batch.Export.image.toDrive(
        image=clean,
        description=f'GHS_POP_{year}_India_clean',
        folder='GEE_Exports',
        fileNamePrefix=f'GHS_POP_{year}_India_clean',
        region=India.geometry(),
        scale=100,          # 100 m — native GHSL resolution
        crs='EPSG:4326',    # WGS84 — India spans UTM zones 42–47
        maxPixels=int(1e10),
        fileFormat='GeoTIFF'
    )
    task.start()
    print(f'Export task submitted: GHS_POP_{year}_India_clean')

print('All export tasks submitted for years:', years)

Export task submitted: GHS_POP_2015_India_clean
Export task submitted: GHS_POP_2020_India_clean
Export task submitted: GHS_POP_2025_India_clean
All export tasks submitted for years: [2015, 2020, 2025]


# GHSL Built-Up Surface Export — India
Export GHS_BUILT_S (P2023A) for India across all available epochs (1975–2020) to Google Drive at 100 m resolution in WGS84.  
Band `built_surface` = m² of built-up surface per 100 m cell.

In [5]:
# ── Loop: export built-up surface for each year ───────────────────────────────
for year in years:
    # 1. Load image
    raw = ee.Image(f'JRC/GHSL/P2023A/GHS_BUILT_S/{year}')

    # 2. Select built-up surface band, clip, clamp negatives to 0
    clean = (
        raw
        .select('built_surface')
        .clip(India)
        .max(ee.Image(0))
    )

    # 3. Export to Drive
    task = ee.batch.Export.image.toDrive(
        image=clean,
        description=f'GHS_BUILT_S_{year}_India_clean',
        folder='GEE_Exports',
        fileNamePrefix=f'GHS_BUILT_S_{year}_India_clean',
        region=India.geometry(),
        scale=100,
        crs='EPSG:4326',
        maxPixels=int(1e10),
        fileFormat='GeoTIFF'
    )
    task.start()
    print(f'Export task submitted: GHS_BUILT_S_{year}_India_clean')

print('All built-up export tasks submitted for years:', years)

Export task submitted: GHS_BUILT_S_2015_India_clean
Export task submitted: GHS_BUILT_S_2020_India_clean
Export task submitted: GHS_BUILT_S_2025_India_clean
All built-up export tasks submitted for years: [2015, 2020, 2025]


# VIIRS Nighttime Lights Export — India (2020–2025)
Annual median composite from NOAA/VIIRS/DNB/MONTHLY_V1/VCMCFG.  
Band `avg_rad` = average radiance (nW/cm²/sr). Native resolution ~500 m.

In [7]:
# ── VIIRS Nighttime Lights: annual median composite ───────────────────────────
ntl_years = list(range(2019, 2020))  # 2020–2025

for year in ntl_years:
    # Monthly VIIRS DNB composites for this calendar year
    monthly = (
        ee.ImageCollection('NOAA/VIIRS/DNB/MONTHLY_V1/VCMCFG')
          .filterDate(f'{year}-01-01', f'{year + 1}-01-01')
          .select('avg_rad')
    )

    # Annual median — suppresses ephemeral light artefacts better than mean
    annual = monthly.median().clip(India).max(ee.Image(0))

    task = ee.batch.Export.image.toDrive(
        image=annual,
        description=f'VIIRS_NTL_{year}_India',
        folder='GEE_Exports',
        fileNamePrefix=f'VIIRS_NTL_{year}_India',
        region=India.geometry(),
        scale=500,
        crs='EPSG:4326',
        maxPixels=int(1e10),
        fileFormat='GeoTIFF'
    )
    task.start()
    print(f'Export task submitted: VIIRS_NTL_{year}_India')

print('All VIIRS NTL export tasks submitted for years:', ntl_years)

Export task submitted: VIIRS_NTL_2019_India
All VIIRS NTL export tasks submitted for years: [2019]
